# YetiRankを用いてcatboost

In [2]:
import pandas as pd, numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier

import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("ok")

ok


In [3]:
# -------------------------
# 1. データ読み込み & サンプリング
# -------------------------
transactions = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/transactions_train.csv")
customers = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/customers.csv")
articles = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/articles.csv")

print("transactions shape: ", transactions.shape)
print("customers shape: ", customers.shape)
print("articles shape: ", articles.shape)

articles.head()

transactions shape:  (31788324, 5)
customers shape:  (1371980, 7)
articles shape:  (105542, 25)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and pro..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and pro..."


In [4]:
print(transactions.columns)
print()
print(customers.columns)
print()
print(articles.columns)

Index(['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'], dtype='object')

Index(['customer_id', 'FN', 'Active', 'club_member_status',
       'fashion_news_frequency', 'age', 'postal_code'],
      dtype='object')

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')


In [5]:
customers.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93f4c830291c32bc3057
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6c9090f7dd3e38380dc
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c222539af5973a23ae6d
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd4564743b005a805b1d


In [6]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and pro..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and pro..."


In [7]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,685687004,0.016932,2


In [8]:
# 顧客側特徴量（例）
customer_features = customers.copy()

# 商品側特徴量（例）
articles_features = articles.copy()

# それらをトランザクションに結合
merged = transactions.merge(customer_features, on="customer_id", how="left")
merged = merged.merge(articles_features, on="article_id", how="left")


In [9]:
# 正例
positive = merged[merged['t_dat'] >= '2020-09-16'][['customer_id', 'article_id']].drop_duplicates()
positive['target'] = 1

# 負例（ランダムサンプリング）
np.random.seed(42)
neg = pd.DataFrame({
    'customer_id': np.random.choice(positive['customer_id'].unique(), size=len(positive)*3),
    'article_id': np.random.choice(positive['article_id'].unique(), size=len(positive)*3)
})
neg['target'] = 0

# 重複除去
neg = neg[~neg.set_index(['customer_id', 'article_id']).index.isin(
    positive.set_index(['customer_id', 'article_id']).index
)]

# 結合
train_df = pd.concat([positive, neg], ignore_index=True)


In [10]:
print(positive.shape)
positive.head()

(213728, 3)


,customer_id,article_id,target
31548013,000fb6e772c5d0023892065e659963da90b1866035558ec16fca51b0dcfb7e59,786022008,1
31548014,000fb6e772c5d0023892065e659963da90b1866035558ec16fca51b0dcfb7e59,913272003,1
31548015,000fb6e772c5d0023892065e659963da90b1866035558ec16fca51b0dcfb7e59,889669006,1
31548016,0010e8eb18f131e724d6997909af0808adbba057529edb1523944f7d4e02b4ce,237347060,1
31548017,0010e8eb18f131e724d6997909af0808adbba057529edb1523944f7d4e02b4ce,562245001,1


In [11]:
print(neg.shape)
neg.head()

(641070, 3)


,customer_id,article_id,target
0,8d8306371b146003fc20b53f6c7d605c61dd0f2a638eaadc73ed6cdfd32391b1,859208005,0
1,17f248684efec32815dfa28c369d8e35daf2fbb752933f76e630cb0fcf8f6010,893994001,0
2,52a547535b48bc4200c279986304fd73a7188d45b43e04b1c6945a9a0f34af9e,806534004,0
3,b6ff61781528abba8d6367854b000adda8de32ccc72a7267e9c083e8b4a0325a,696991027,0
4,5b8904833fefde61f2c4b61a8ca067e12603cd681deaa5c2329279f26f27c27c,858052006,0


In [12]:
# グループセッション定義
train_df['group_id'] = train_df['customer_id']

# group_id順に並べ替え（これが必要！）
train_df = train_df.sort_values('group_id').reset_index(drop=True)

In [13]:
print(train_df.shape)
train_df.head()

(854798, 4)


,customer_id,article_id,target,group_id
0,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793,864295002,0,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793
1,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793,816832012,0,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793
2,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793,189616006,0,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793
3,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793,679854018,0,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793
4,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793,624486001,1,00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793


In [14]:
print(train_df.columns)

Index(['customer_id', 'article_id', 'target', 'group_id'], dtype='object')


In [15]:
from catboost import CatBoostRanker, Pool

# 特徴量の結合（ここでは例として商品属性）
train_df = train_df.merge(articles_features, on='article_id', how='left')
train_df = train_df.merge(customer_features, on='customer_id', how='left')

# 説明変数
feature_cols = ['product_type_no', 'index_group_no', 'age']  # 必要に応じて拡張
cat_cols = ['product_type_no', 'index_group_no']

# Poolの作成（YetiRank用）
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df['target'],
    group_id=train_df['group_id'],
    cat_features=cat_cols
)

# モデルの学習
model = CatBoostRanker(
    iterations=300,
    learning_rate=0.1,
    loss_function='YetiRank',
    eval_metric='NDCG',
    verbose=50
)

model.fit(train_pool)


Groupwise loss function. OneHotMaxSize set to 10
0:	total: 330ms	remaining: 1m 38s
50:	total: 11.9s	remaining: 58.1s
100:	total: 22.9s	remaining: 45.2s
150:	total: 33.8s	remaining: 33.3s
200:	total: 45.2s	remaining: 22.2s
250:	total: 56.4s	remaining: 11s
299:	total: 1m 7s	remaining: 0us


In [ ]:
"""
↓メモリクラッシュする
# 推薦対象ペアを作成
test_customers = train_df['customer_id'].unique()
test_articles = articles['article_id'].unique()

# 全組み合わせを作成（メモリ注意）
test_df = pd.DataFrame([(c, a) for c in test_customers for a in test_articles],
                       columns=['customer_id', 'article_id'])

# 特徴量結合（学習と同じ）
test_df = test_df.merge(articles_features, on='article_id', how='left')
test_df = test_df.merge(customer_features, on='customer_id', how='left')

# 予測
test_pool = Pool(test_df[feature_cols], cat_features=cat_cols)
test_df['score'] = model.predict(test_pool)

# 顧客ごとに12件上位を抽出
recommendations = test_df.sort_values(['customer_id', 'score'], ascending=[True, False]) \
                         .groupby('customer_id')['article_id'] \
                         .apply(lambda x: x.head(12).tolist())
"""

# 1. 人気商品1000件を使う
popular_articles_1000 = (
    transactions['article_id']
    .value_counts()
    .head(1000)
    .index
    .tolist()
)

# 全体人気上位（補完用に12件だけ抽出）
popular_articles = [str(a).zfill(10) for a in transactions['article_id'].value_counts().head(12).index.tolist()]

test_customers = train_df['customer_id'].unique()
test_articles = popular_articles_1000  # 商品を1000件に絞る！

# 2. 全組み合わせを作成（1000万行未満に抑える）
test_df = pd.DataFrame([(c, a) for c in test_customers for a in test_articles],
                       columns=['customer_id', 'article_id'])

# 3. 特徴量結合
test_df = test_df.merge(articles_features, on='article_id', how='left')
test_df = test_df.merge(customer_features, on='customer_id', how='left')

# 4. CatBoostによる予測
test_pool = Pool(test_df[feature_cols], cat_features=cat_cols)
test_df['score'] = model.predict(test_pool)

# 5. CatBoost予測に基づいて、顧客ごとに上位12件を抽出
recommendations = test_df.sort_values(['customer_id', 'score'], ascending=[True, False]) \
                         .groupby('customer_id')['article_id'] \
                         .apply(lambda x: x.head(12).tolist())

# 🔽 推薦リスト内の article_id をゼロ埋め（10桁）
recommendations = recommendations.apply(lambda x: [str(a).zfill(10) for a in x])

# --- 🔧 unknown_customers を作成（test_customers + 年齢層 + 最新購入日） ---
# 顧客マスタから年齢層を取得
customer_info = customers[['customer_id', 'age']].copy()
customer_info['age_group3'] = pd.cut(customer_info['age'], bins=[0, 25, 50, 100], labels=['young', 'middle', 'senior'])

# 各顧客の最新購入日を transactions から取得（時系列の推定用）
latest_dates = transactions.groupby('customer_id')['t_dat'].max().reset_index()
latest_dates['t_dat'] = pd.to_datetime(latest_dates['t_dat'])

# test_customers から unknown_customers を構築
unknown_customers = pd.DataFrame({'customer_id': test_customers})
unknown_customers = unknown_customers.merge(customer_info, on='customer_id', how='left')
unknown_customers = unknown_customers.merge(latest_dates, on='customer_id', how='left')


# --- 🔽 ここから補完処理を追加：時系列 × 年齢層で12件に保証 ---
# 顧客の時期（四半期）を生成
unknown_customers["quarter"] = pd.PeriodIndex(unknown_customers["t_dat"], freq="Q").astype(str)

# 顧客属性とマージ
merged_pred = recommendations.reset_index().merge(
    unknown_customers[["customer_id", "quarter", "age_group3"]],
    on="customer_id",
    how="left"
)

# 補完関数の定義
def recommend_12_articles_with_context(article_list, quarter, age_group):
    if len(article_list) >= 12:
        return article_list[:12]
    else:
        needed = 12 - len(article_list)
        key = (quarter, age_group)
        if key in top_items_dict:
            fill_items = [a for a in top_items_dict[key] if a not in article_list]
        else:
            fill_items = [a for a in popular_articles if a not in article_list]
        return article_list + fill_items[:needed]

# 補完処理の適用
merged_pred['prediction'] = merged_pred.apply(
    lambda row: recommend_12_articles_with_context(row['article_id'], row['quarter'], row['age_group3']),
    axis=1
)

# --- 🔽 ここから元の提出整形処理に合流 ---

# 6. 提出形式に変換（スペース区切り）
merged_pred['prediction'] = merged_pred['prediction'].apply(lambda x: ' '.join(x))
recommendations_df = merged_pred[['customer_id', 'prediction']]
recommendations_df['customer_id'] = recommendations_df['customer_id'].astype(str)

# 7. sample_submission にマージ（提出形式に合わせる）
sample_submission = pd.read_csv('/Users/kurokawa/Desktop/kaggle/H&M/sample_submission.csv')
submission = sample_submission[['customer_id']].merge(recommendations_df, on='customer_id', how='left')

# fallback（全体人気12件）を使って欠損を埋める
fallback_12 = ' '.join(popular_articles[:12])
submission['prediction'] = submission['prediction'].fillna(fallback_12)

# 8. CSV出力
times = 2
csv_name = f"YetiRank_catboost_{times}.csv"
submission.to_csv(csv_name, index=False)
print(f"✅ {csv_name} を出力しました:")
submission.head()


✅ YetiRank_catboost_2.csv を出力しました:
                                                        customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a   

                                                                                            prediction  
0  0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...  
1  0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...  
2  0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...  
3  0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...  
4  0706016001

In [23]:
submission.head()

,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a,0706016001 0706016002 0372860001 0610776002 0759871002 0464297007 0372860002 0610776001 03992230...
